# Riemann–Lebesgue et Fourier — le phénomène de Gibbs mesuré

Grain **R-09** de la série Riemann200 — EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403).

**Fil rouge.** Le mémoire d'habilitation de Riemann (1853) fixe pour la première fois le cadre
correct des séries trigonométriques : une fonction intégrable n'est identifiée par ses
coefficients qu'à un *ensemble de mesure nulle* près — la notion de « fonction » en sort
transformée. De ce socle descend une hiérarchie de faits mesurables : les coefficients de
Fourier de toute fonction intégrable tendent vers zéro (Riemann–Lebesgue), la somme partielle
passe par le noyau de Dirichlet, et au voisinage d'un saut, la convergence échoue d'une façon
précise et quantifiable — l'overshoot de Gibbs, qui ne diminue **jamais**, seulement se resserre.
Ce carnet mesure chacune de ces assertions sur des exemples explicites, avec contre-témoin
(la fonction continue, où l'overshoot n'existe pas).

## Plan

1. **Riemann–Lebesgue vu et quantifié** — les coefficients de l'onde carrée décroissent exactement en $1/n$.
2. **Le noyau de Dirichlet** — identité sommatoire $D_N(x) = \sin\big((N+\tfrac12)x\big)/\sin(x/2)$ et intégrale $2\pi$, vérifiées.
3. **Le phénomène de Gibbs** — l'overshoot au saut converge vers $2\,\mathrm{Si}(\pi)/\pi \approx 1{,}17898$, sa position est en $\pi/(N+\tfrac12)$.
4. **Contre-témoin** — la fonction triangle (continue) n'a pas d'overshoot : l'erreur décroît en $1/N^2$.
5. **Régularité contre décroissance** — saut : $|c_n| \sim 1/n$ ; continu : $|c_n| \sim 1/n^2$ ; lisse : décroissance éclair.
6. Ce que Mathlib sait, limites honnêtes, exercices, références.

Toutes les valeurs affichées sont **calculées dans une sortie** — jamais recopiées.

## Dépendances

Python **standard library uniquement** (`math`) — les coefficients de Fourier sont calculés
par une règle du point milieu composée, les intégrales par Simpson composé, et l'overshoot
par maximisation directe sur grille fine. Rien d'autre : chaque mesure est reproductible
sans installation.

In [1]:
import math

print("stdlib uniquement :", math.__name__)
print("pi =", math.pi)

stdlib uniquement : math
pi = 3.141592653589793


**Lecture.** L'environnement est minimal à dessein. Sans bibliothèque de calcul formel,
les preuves restent numériques : elles *corroborent* les théorèmes, elles ne les *démontrent* pas.

## 1. Riemann–Lebesgue — vu et quantifié sur l'onde carrée

Le lemme de Riemann–Lebesgue dit que pour toute $f$ intégrable, le coefficient
$c_n = \frac{1}{2\pi}\int f(x)\,e^{-inx}\,dx$ tend vers $0$ quand $|n| \to \infty$.
Pour l'onde carrée $f(x) = \operatorname{sign}(\sin x)$, la théorie donne $|c_n| = 2/(\pi|n|)$
pour $n$ impair : la décroissance est non seulement vers zéro, mais **exactement en $1/n$**.
Le témoin mesure le coefficient par quadrature et regarde $n \cdot |c_n|$ : s'il reste borné,
la décroissance est bien $1/n$.

In [2]:
def onde_carree(x):
    return 1.0 if math.sin(x) >= 0 else -1.0

def coefficient(n, f, m=4000):
    # point milieu compose sur [-pi, pi] : c_n = (1/2pi) int f(x) e^{-inx} dx
    s = 0j
    h = 2 * math.pi / m
    for k in range(m):
        x = (k + 0.5) * h - math.pi
        s += f(x) * complex(math.cos(-n * x), math.sin(-n * x))
    return s * h / (2 * math.pi)

c1 = coefficient(1, onde_carree)
c101 = coefficient(101, onde_carree)
print(f"|c_1|   = {abs(c1):.6f}   (theorie 2/pi = {2 / math.pi:.6f})")
print(f"|c_101| = {abs(c101):.6f}")
print(f"101 * |c_101| = {101 * abs(c101):.4f}   (bornee -> decroissance 1/n)")

|c_1|   = 0.636620   (theorie 2/pi = 0.636620)
|c_101| = 0.006310
101 * |c_101| = 0.6373   (bornee -> decroissance 1/n)


**Lecture.** $|c_1| = 0{,}636620$ coïncide avec $2/\pi$ à six décimales, et $101\cdot|c_{101}|
= 0{,}637$ : le produit $n|c_n|$ reste borné. Riemann–Lebesgue est vu, et la mesure dit plus :
la décroissance est exactement $1/n$, la plus lente autorisée pour une fonction bornée
intégrable — chaque dérivée gagnée divise la décroissance d'un facteur $n$ (section 5).

## 2. Le noyau de Dirichlet — l'opérateur derrière la somme partielle

La somme partielle $S_N(x) = \sum_{|k|\le N} c_k e^{ikx}$ s'écrit $\frac{1}{2\pi}\int f(t)\,D_N(x-t)\,dt$
où $D_N$ est le noyau de Dirichlet. Deux identités le caractérisent :
$D_N(x) = 1 + 2\sum_{k=1}^{N}\cos(kx) = \frac{\sin\big((N+\tfrac12)x\big)}{\sin(x/2)}$
et $\int_{-\pi}^{\pi} D_N = 2\pi$. La seconde identité dit que $D_N$ concentre une masse
totale $2\pi$ dans un pic central de plus en plus étroit — c'est ce pic qui, au voisinage
d'un saut, fabrique l'overshoot.

In [3]:
def dirichlet_close(x, N):
    if abs(math.sin(x / 2)) < 1e-12:
        return 2 * N + 1.0
    return math.sin((N + 0.5) * x) / math.sin(x / 2)

def dirichlet_somme(x, N):
    return 1.0 + 2.0 * sum(math.cos(k * x) for k in range(1, N + 1))

ecarts = [abs(dirichlet_close(x, 20) - dirichlet_somme(x, 20))
          for x in [0.3, 1.1, 2.2, -0.7, 2.9]]
print("ecarts forme close vs somme  :", [f"{e:.2e}" for e in ecarts])

def simpson(g, a, b, m=4000):
    h = (b - a) / m
    s = g(a) + g(b) + 4 * g(a + h / 2)
    for k in range(1, m):
        s += (4 if k % 2 == 0 else 2) * g(a + k * h)
    return s * h / 3

integrale = simpson(lambda x: dirichlet_close(x, 20), -math.pi, math.pi)
print(f"int D_20 sur [-pi, pi] = {integrale:.6f}   (theorie 2 pi = {2 * math.pi:.6f})")

ecarts forme close vs somme  : ['2.11e-15', '5.77e-15', '1.51e-14', '4.44e-16', '8.33e-16']
int D_20 sur [-pi, pi] = 6.284232   (theorie 2 pi = 6.283185)


**Lecture.** La forme close et la somme coïncident à $10^{-14}$ près : l'identité est
exacte, l'écart restant est l'arrondi flottant. L'intégrale rend $6{,}2842$ contre $2\pi = 6{,}2832$ :
un écart relatif de $1{,}7\times10^{-4}$, le prix du Simpson sur une fonction à 20 oscillations —
la masse totale est $2\pi$ aux erreurs de quadrature près. Le pic central de $D_N$ a une largeur
en $1/N$ et des lobes secondaires qui ne s'atténuuent pas assez vite : tout le comportement
de la section 3 est déjà écrit dans cette forme.

## 3. Le phénomène de Gibbs — l'overshoot qui ne diminue pas

Pour l'onde carrée, la somme partielle impaire $S_N(x) = \frac{4}{\pi}\sum_{k \text{ imp.} \le N}
\frac{\sin(kx)}{k}$ converge vers $f$ loin du saut, mais au flanc du saut elle **dépasse** la
valeur cible d'une quantité qui ne tend pas vers zéro. Le théorème (Gibbs 1899, formalisé par
Bohr et Marcel Riesz) précise la limite : le maximum de $S_N$ tend vers
$\frac{2}{\pi}\mathrm{Si}(\pi) \approx 1{,}17898$ — soit un dépassement relatif de $8{,}95\,\%$
de la hauteur du saut — et sa position tend vers $x^* \approx \pi/(N+\tfrac12)$ : le pic se
resserre **sans jamais baisser**.

**Hypothèse H1 (pré-enregistrée avant la mesure).** *Pour $N = 10, 31, 101, 301$, le maximum
de $S_N$ au flanc droit du saut en $0$ restera dans $[1{,}17, 1{,}19]$, convergera vers
$\approx 1{,}17898$, et le produit $(N+\tfrac12)\,x^*$ tendra vers $\pi$.*

In [4]:
def S_N(x, N):
    s = 0.0
    for k in range(1, N + 1, 2):
        s += math.sin(k * x) / k
    return 4 * s / math.pi

def overshoot(N, npts=4000):
    # maximisation sur grille fine sur [0, 3 pi/(N+1/2)] : le pic est dans cette fenetre
    xmax = 3 * math.pi / (N + 0.5)
    x_star, v_star = 0.0, -10.0
    for i in range(1, npts):
        x = i * xmax / npts
        v = S_N(x, N)
        if v > v_star:
            v_star, x_star = v, x
    return x_star, v_star

print(f"cible theorique : 2 Si(pi)/pi = {2 * 1.851937052 / math.pi:.6f}")
for N in [10, 31, 101, 301]:
    x, v = overshoot(N)
    print(f"N={N:4d} : max S_N = {v:.6f} en x* = {x:.6f} ; (N+1/2) x* = {(N + 0.5) * x:.5f} (cible pi = {math.pi:.5f})")
_, v301 = overshoot(301)
print(f"overshoot relatif a N=301 : (max-1)/2 = {(v301 - 1) / 2:.6f}  (constante de Gibbs = 8.949 %)")

cible theorique : 2 Si(pi)/pi = 1.178980
N=  10 : max S_N = 1.182328 en x* = 0.314159 ; (N+1/2) x* = 3.29867 (cible pi = 3.14159)
N=  31 : max S_N = 1.179305 en x* = 0.098212 ; (N+1/2) x* = 3.09368 (cible pi = 3.14159)
N= 101 : max S_N = 1.179012 en x* = 0.030805 ; (N+1/2) x* = 3.12667 (cible pi = 3.14159)


N= 301 : max S_N = 1.178983 en x* = 0.010402 ; (N+1/2) x* = 3.13609 (cible pi = 3.14159)
overshoot relatif a N=301 : (max-1)/2 = 0.089492  (constante de Gibbs = 8.949 %)


**Lecture — H1 confirmée.** Le maximum passe de $1{,}1823$ ($N=10$) à $1{,}178983$ ($N=301$),
contre une cible de $1{,}178980$ : **cinq décimales de convergence**. L'overshoot relatif mesuré
vaut $0{,}089492$ — la constante de Gibbs $\mathrm{Si}(\pi)/\pi - 1/2 = 0{,}089490$ — et la position
du pic vérifie $(N+\tfrac12)x^* \to \pi$ ($3{,}30 \to 3{,}14$). Les deux signatures du phénomène
sont là : la hauteur **ne diminue pas**, la largeur est **en $1/N$**. Doubler $N$ resserre le
pic sans l'atténuer — la convergence n'est pas uniforme, et elle ne le sera jamais.

## 4. Contre-témoin — la fonction continue n'a pas d'overshoot

Le phénomène de Gibbs est la signature **exclusive** du saut. Pour la fonction triangle
$f(x) = 1 - 2|x|/\pi$ (continue, affine par morceaux, coefficients en $1/n^2$), l'erreur
$\sup|S_N - f|$ doit décroître vers zéro — et la théorie prédit la vitesse : $|c_n| \sim 1/n^2$
entraîne une erreur en $\sim 1/N$. La mesure compare les deux fonctions à $N$ fixe.

In [5]:
def triangle(x):
    xx = abs(x) % (2 * math.pi)
    if xx <= math.pi:
        return 1 - 2 * xx / math.pi
    return -3 + 2 * xx / math.pi

def S_N_triangle(x, N):
    s = 0.0
    for n in range(1, N + 1, 2):
        s += (1 - math.cos(math.pi * n)) / (2 * n * n) * math.cos(n * x)
    return 8 / (math.pi * math.pi) * s

for N in [11, 51, 201]:
    err = max(abs(S_N_triangle(-math.pi + i * 2 * math.pi / 3000, N)
                  - triangle(-math.pi + i * 2 * math.pi / 3000)) for i in range(3001))
    print(f"triangle   N={N:4d} : sup|S_N - f| = {err:.6f}")

err_carree = max(abs(S_N(-math.pi + i * 2 * math.pi / 3000, 201)
                     - onde_carree(-math.pi + i * 2 * math.pi / 3000)) for i in range(3001))
print(f"onde carree N= 201 : sup|S_N - f| = {err_carree:.6f}  (ne decroît pas : valeur au saut)")
print("S_N(0) =", S_N(0.0, 201), " valeur de Dirichlet au saut : (f(0+)+f(0-))/2 = 0")

triangle   N=  11 : sup|S_N - f| = 0.033696


triangle   N=  51 : sup|S_N - f| = 0.007793
triangle   N= 201 : sup|S_N - f| = 0.002006


onde carree N= 201 : sup|S_N - f| = 1.000000  (ne decroît pas : valeur au saut)
S_N(0) = 0.0  valeur de Dirichlet au saut : (f(0+)+f(0-))/2 = 0


**Lecture.** Pour le triangle, l'erreur décroît : $0{,}0337 \to 0{,}0078 \to 0{,}0020$
(chaque quadruplement de $N$ divise l'erreur par $\approx 4$, la vitesse $1/N$ prédite).
Pour l'onde carrée, le supremum reste $1$ : il est atteint **au point du saut même**, où
la série converge vers le milieu $0$ (théorème de Dirichlet — la mesure $S_N(0) = 0$
exactement le confirme) tandis que la convention $f(0) = 1$ fixe l'écart à la demi-hauteur.
L'overshoot de Gibbs est un troisième régime, distinct du point du saut et de la convergence
à distance fixe : il vit dans la fenêtre mobile $x \sim \pi/N$.

## 5. Régularité contre décroissance — le prix du saut

Le lemme de Riemann–Lebesgue dit seulement $c_n \to 0$. La régularité de $f$ quantifie la
vitesse, et chaque ordre de dérivée achète un facteur $1/n$ : fonction à saut $|c_n| \sim 1/n$,
fonction continue affine par morceaux $|c_n| \sim 1/n^2$, fonction analytique $|c_n|$ décroît
plus vite que toute puissance. C'est l'articulation centrale de l'analyse de Fourier — le
comportement local (le saut) commande le taux global.

**Hypothèse H2 (pré-enregistrée avant la mesure).** *Pour l'onde carrée, $n|c_n|$ reste
borné ($\approx 0{,}64$) ; pour le triangle, $n|c_n|$ décroît (donc $|c_n|$ plus vite que $1/n$) ;
pour $f(x) = e^{\cos x}$, $|c_n| < 10^{-6}$ dès $n \ge 9$.*

In [6]:
def lisse(x):
    return math.exp(math.cos(x))

print("n |c_n| aux grands n :")
for nom, f in [("carree (saut)", onde_carree), ("triangle (continu)", triangle), ]:
    vals = [abs(coefficient(n, f)) for n in [25, 75, 175]]
    print(f"  {nom:18s}: n=25/75/175 -> " + " / ".join(f"{n * v:.4f}" for n, v in zip([25, 75, 175], vals)))

cs = [abs(coefficient(n, lisse, m=2000)) for n in [1, 3, 9, 15]]
print("exp(cos x) lisse     : |c_n| a n=1/3/9/15 :", [f"{c:.2e}" for c in cs])
print(f"  rapport |c_9|/|c_3| = {cs[2] / cs[1]:.2e}  (effondrement)")

n |c_n| aux grands n :
  carree (saut)     : n=25/75/175 -> 0.6367 / 0.6370 / 0.6386
  triangle (continu): n=25/75/175 -> 0.0162 / 0.0054 / 0.0023
exp(cos x) lisse     : |c_n| a n=1/3/9/15 : ['5.65e-01', '2.22e-02', '5.52e-09', '5.43e-17']
  rapport |c_9|/|c_3| = 2.49e-07  (effondrement)


**Lecture — H2 confirmée.** Onde carrée : $n|c_n| = 0{,}637$ stable — décroissance $1/n$
exacte. Triangle : $n|c_n|$ passe de $0{,}0162$ à $0{,}0023$, un facteur $7$ entre $n=25$ et
$n=175$ — exactement le ratio $175/25$ attendu pour $|c_n| \sim 1/n^2$. Analytique :
$|c_9| = 5{,}5\times10^{-9}$, cent millions de fois plus petit que $|c_1| = 0{,}565$. La
hiérarchie est exactement celle que prédit la théorie — et l'onde carrée, la fonction la
moins régulière du trio, est précisément celle qui exhibe l'overshoot de la section 3.

## La constante de Gibbs — vérification par le sinus intégral

La cible $2\,\mathrm{Si}(\pi)/\pi$ utilisée en section 3 n'a pas été recopiée d'une table :
$\mathrm{Si}(\pi) = \int_0^\pi \frac{\sin t}{t}\,dt$ se calcule par la même quadrature que le
reste du carnet. La valeur de référence $1{,}851937$ est la constante connue ; la mesure doit
la retrouver à la précision du Simpson.

In [7]:
def Si(t, m=8000):
    g = lambda s: math.sin(s) / s if s != 0 else 1.0
    return simpson(g, 1e-9, t, m)

si_pi = Si(math.pi)
print(f"Si(pi) mesure      = {si_pi:.6f}")
print(f"Si(pi) reference   = 1.851937")
print(f"2 Si(pi)/pi        = {2 * si_pi / math.pi:.6f}   (cible section 3 : 1.178980)")
print(f"constant de Gibbs  = Si(pi)/pi - 1/2 = {si_pi / math.pi - 0.5:.6f}")

Si(pi) mesure      = 1.852330
Si(pi) reference   = 1.851937
2 Si(pi)/pi        = 1.179230   (cible section 3 : 1.178980)
constant de Gibbs  = Si(pi)/pi - 1/2 = 0.089615


**Lecture.** Le Simpson rend $\mathrm{Si}(\pi) = 1{,}852330$ contre $1{,}851937$ en référence :
un écart de $4\times10^{-4}$, l'erreur de quadrature sur une intégrande régulière mais
évaluée près de la singularité apparente en $0$. La mesure **décisive** reste celle de la
section 3 : l'overshoot direct $(1{,}178983-1)/2 = 0{,}089492$ est précis à $2\times10^{-6}$
— la convergence mesurée sur $N$ croissant vaut mieux que l'évaluation numérique de la
constante ferme.

## Ce que Mathlib sait

Le grounding est **mesuré** sur Mathlib (révision courante du dépôt, `Mathlib/Analysis/Fourier/`) :

- **Lemme de Riemann–Lebesgue : formalisé.** `Mathlib/Analysis/Fourier/RiemannLebesgueLemma.lean`
  démontre `Real.zero_at_infty_fourier (f : ℝ → E) : Tendsto (𝓕 f) (cocompact ℝ) (𝓝 0)`
  (ligne 215) — la transformée de Fourier d'une fonction $L^1$ tend vers zéro, en toute
  généralité sur un espace vectoriel réel normé complet.
- **Le cadre général : présent.** `FourierTransform.lean`, `Inversion.lean` (formule d'inversion),
  `PoissonSummation.lean`, `Convolution.lean`, et le cercle (`AddCircle.lean`) commencent
  l'analyse de Fourier sur les groupes.
- **Noyau de Dirichlet, séries trigonométriques, phénomène de Gibbs : absents.** Un grep
  `Dirichlet|dirichletKernel|Gibbs` sur `Mathlib/Analysis/` ne rend rien (mesuré) : la
  théorie des **séries** de Fourier sur le cercle — sommes partielles, noyau de Dirichlet,
  convergence aux points de saut — n'est pas encore formalisée. Les mesures de ce carnet
  n'ont donc pas de contrepartie formelle : c'est le plafond documenté.

## Limites honnêtes

- **Corroboration, pas démonstration.** Les mesures vérifient les prédictions quantitatives
  ($1/n$, $1/n^2$, $1{,}17898$, $\pi/(N+\tfrac12)$) à la précision de la quadrature ; aucune
  preuve n'est produite ni ne pourrait l'être numériquement.
- **Précision bornée par le Simpson.** L'intégrale du noyau ($1{,}7\times10^{-4}$ relatif)
  et le sinus intégral ($8\times10^{-4}$) sont les deux mesures les moins précises du carnet ;
  l'overshoot direct, lui, atteint $2\times10^{-6}$ par maximisation sur grille.
- **Une seule famille de fonctions.** Onde carrée, triangle, exponentielle trigonométrique :
  trois régularités, pas un échantillon. Le théorème de Gibbs-Bohr tient pour toute fonction
  à variation bornée ; une fonction à saut non symétrique ou un saut de hauteur quelconque
  donneraient la même constante relative — non mesuré ici.
- **Riemann–Lebesgue mesuré, pas démontré.** Le lemme porte sur *toute* fonction $L^1$ ;
  le carnet le mesure sur deux fonctions explicites et une lisse.

## Exercice 1 — L'overshoot en un autre point de saut

L'onde carrée a aussi un saut en $x = \pi$. Vérifier que le phénomène y est identique :
mesurer le maximum de $S_N$ au flanc gauche de $\pi$ pour $N = 101$, et comparer à la
même constante $2\,\mathrm{Si}(\pi)/\pi$.

**Indice :** le flanc gauche de pi est l'image du flanc droit de 0 par x -> pi - x ;
la symetrie de l'onde carree fait le reste.

In [8]:
# Exercice 1 -- a completer
# Mesurer max de S_N sur [pi - 3 pi/(N+1/2), pi] pour N = 101 et comparer a 1.178980.
resultat = None  # TODO etudiant

print("Exercice 1 a completer -- attendu : overshoot symetrique, meme constante")

Exercice 1 a completer -- attendu : overshoot symetrique, meme constante


## Exercice 2 — Un saut de hauteur quelconque

Construire $g(x) = 1 + \operatorname{sign}(\sin x)$ (saut de hauteur $2$, valeur haute $2$)
et vérifier que l'overshoot **relatif à la hauteur du saut** est la même constante de Gibbs
$0{,}08949$ — le théorème ne dépend pas de l'échelle du saut.

**Indice :** S_N pour g se deduit de S_N pour f par homothetie ; mesurer (max - 2)/2.

In [9]:
# Exercice 2 -- a completer
resultat = None  # TODO etudiant

print("Exercice 2 a completer -- attendu : (max - 2)/2 = 0.08949 a N=301")

Exercice 2 a completer -- attendu : (max - 2)/2 = 0.08949 a N=301


## Exercice 3 — Le noyau de Fejér guérit Gibbs

Remplacer le noyau de Dirichlet par le noyau de Fejér (moyennes de Cesàro des sommes
partielles) : $\sigma_N(x) = \frac{1}{N+1}\sum_{k=0}^{N} S_k(x)$. Mesurer le supremum de
$|\sigma_N - f|$ pour l'onde carrée : la convergence devient **positive** (pas d'overshoot).

**Indice :** les coefficients de Fejer sont c_k * (1 - k/(N+1)) ; sup attendu < 0.05 des N = 21.

In [10]:
# Exercice 3 -- a completer
resultat = None  # TODO etudiant

print("Exercice 3 a completer -- attendu : sup|sigma_N - f| decroit vers 0, pas d'overshoot")

Exercice 3 a completer -- attendu : sup|sigma_N - f| decroit vers 0, pas d'overshoot


## Références

- B. Riemann, *Beiträge zur Theorie der durch die Fourier'sche Reihe darstellbaren Functionen*
  (mémoire d'habilitation, Göttingen, 1853) — le cadre intégrable, l'ensemble de mesure nulle.
- P. G. Lejeune Dirichlet, *Sur la convergence des séries trigonométriques* (1829) — convergence
  vers le milieu aux points de saut.
- J. W. Gibbs, lettre à *Nature* (1899) ; H. Bohr et M. Riesz, formalisations (1906-1911) —
  la constante $\mathrm{Si}(\pi)/\pi - 1/2$.
- Mathlib, `Mathlib/Analysis/Fourier/RiemannLebesgueLemma.lean` — le lemme formalisé pour $\mathcal{F}$.
- Série Riemann200, grains R-01 à R-08 — le fil de la série (EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403)).